# 05 — Candidate Fusion + Learning-to-Rank

This notebook combines candidate generators and trains a ranking model.

Architecture:

```text
Co-visitation ─┐
Item2Vec ──────┼── candidate union ── features ── LambdaMART ── Top-K
BPR ───────────┤
Two-Tower ─────┘
```

Retrieval maximizes candidate recall. Ranking then uses richer features to order those candidates.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 2. Load processed data

We reuse the same temporal validation examples created in Notebook 02.


In [ ]:
event_files = sorted(PROCESSED_DIR.glob("otto_train_*_sessions.parquet"))
valid_files = sorted(PROCESSED_DIR.glob("otto_validation_*_sessions.parquet"))

if not event_files or not valid_files:
    raise FileNotFoundError("Run Notebook 02 first.")

events = pd.read_parquet(event_files[-1])
valid_examples = pd.read_parquet(valid_files[-1])

print(events.shape)
print(valid_examples.shape)


## 3. Build item-level ranking features

The ranker can use signals that would be too expensive or unnecessary during full-catalog retrieval.

Examples:
- item popularity
- click/cart/order counts
- conversion proxies
- whether candidate appeared earlier in the session
- rank from each retrieval model


In [ ]:
item_stats = (
    events.groupby("aid")
    .agg(
        item_interactions=("aid", "size"),
        item_last_ts=("ts", "max"),
    )
)

event_counts = (
    events.groupby(["aid", "type"])
    .size()
    .unstack(fill_value=0)
)

item_stats = item_stats.join(
    event_counts,
    how="left",
).fillna(0)

for col in ["clicks", "carts", "orders"]:
    if col not in item_stats.columns:
        item_stats[col] = 0

item_stats["cart_rate"] = (
    item_stats["carts"]
    / item_stats["clicks"].clip(lower=1)
)

item_stats["order_rate"] = (
    item_stats["orders"]
    / item_stats["clicks"].clip(lower=1)
)

item_stats.head()


## 4. Candidate generator interface

This notebook expects recommendation functions from prior notebooks, for example:

```python
recommend_covisit(row, k=100)
recommend_item2vec(row, k=100)
recommend_bpr(row, k=100)
recommend_two_tower(row, k=100)
```

Run those notebooks first in the same kernel, or later refactor the functions into `src/recommendation/`.


In [ ]:
def available_candidate_generators():
    generators = {}

    if "recommend_covisit" in globals():
        generators["covisit"] = recommend_covisit

    if "recommend_item2vec" in globals():
        generators["item2vec"] = recommend_item2vec

    if "recommend_bpr" in globals():
        generators["bpr"] = recommend_bpr

    if "recommend_two_tower" in globals():
        generators["two_tower"] = recommend_two_tower

    return generators


candidate_generators = available_candidate_generators()

print(
    "Available generators:",
    list(candidate_generators.keys())
)

if not candidate_generators:
    raise RuntimeError(
        "Load at least one candidate generator before building ranking data."
    )


## 5. Candidate union

Each retriever contributes candidates and its rank becomes a feature.

Example:

```text
candidate 123:
covisit_rank=3
item2vec_rank=17
two_tower_rank=5
```

Agreement across retrieval models is often a strong ranking signal.


In [ ]:
CANDIDATES_PER_GENERATOR = 50
MAX_CANDIDATES = 200

def generate_candidate_union(row):
    candidate_sources = {}

    for source_name, recommend_fn in candidate_generators.items():
        recommendations = recommend_fn(
            row,
            k=CANDIDATES_PER_GENERATOR,
        )

        for rank, aid in enumerate(recommendations, start=1):
            aid = int(aid)

            candidate_sources.setdefault(
                aid,
                {},
            )

            candidate_sources[aid][
                f"{source_name}_rank"
            ] = rank

    # Bound ranking workload.
    return dict(
        list(candidate_sources.items())[
            :MAX_CANDIDATES
        ]
    )


## 6. Candidate feature builder

In [ ]:
def build_candidate_features(
    row,
    candidate_aid,
    source_features,
):
    observed_set = set(row.observed_aids)

    features = {
        "session_length": len(row.observed_aids),
        "candidate_seen_before": int(
            candidate_aid in observed_set
        ),
        "item_interactions": 0.0,
        "item_clicks": 0.0,
        "item_carts": 0.0,
        "item_orders": 0.0,
        "item_cart_rate": 0.0,
        "item_order_rate": 0.0,
    }

    if candidate_aid in item_stats.index:
        stats = item_stats.loc[candidate_aid]

        features.update({
            "item_interactions": float(
                stats["item_interactions"]
            ),
            "item_clicks": float(stats["clicks"]),
            "item_carts": float(stats["carts"]),
            "item_orders": float(stats["orders"]),
            "item_cart_rate": float(
                stats["cart_rate"]
            ),
            "item_order_rate": float(
                stats["order_rate"]
            ),
        })

    # Encode each retrieval model's rank.
    # Missing candidate => rank 999 and reciprocal-rank 0.
    for source in [
        "covisit",
        "item2vec",
        "bpr",
        "two_tower",
    ]:
        rank = source_features.get(
            f"{source}_rank"
        )

        features[f"{source}_rank"] = (
            float(rank)
            if rank is not None
            else 999.0
        )

        features[f"{source}_rr"] = (
            1.0 / rank
            if rank is not None
            else 0.0
        )

    return features


## 7. Build ranking rows

For every session we create one row per candidate.

`label=1` means the candidate is the held-out target.

For ranker training only, the target is explicitly inserted if retrieval missed it. Candidate recall must still be evaluated separately, because production cannot rank an item that retrieval never produced.


In [ ]:
MAX_RANKING_SESSIONS = 10_000

ranking_rows = []

for row in valid_examples.head(
    MAX_RANKING_SESSIONS
).itertuples(index=False):

    candidate_map = generate_candidate_union(
        row
    )

    # Oracle insertion is training-only.
    candidate_map.setdefault(
        int(row.target_aid),
        {},
    )

    for candidate_aid, source_features in candidate_map.items():
        features = build_candidate_features(
            row,
            candidate_aid,
            source_features,
        )

        ranking_rows.append({
            "session": row.session,
            "candidate_aid": candidate_aid,
            "label": int(
                candidate_aid
                == row.target_aid
            ),
            **features,
        })

ranking_df = pd.DataFrame(
    ranking_rows
)

print(ranking_df.shape)
print(
    "Positive rate:",
    ranking_df["label"].mean()
)

ranking_df.head()


## 8. Split by session, not by row

Rows from the same ranking query must stay together.


In [ ]:
ranking_sessions = (
    ranking_df["session"]
    .drop_duplicates()
    .to_numpy()
)

split_idx = int(
    len(ranking_sessions) * 0.8
)

train_session_ids = set(
    ranking_sessions[:split_idx]
)

valid_session_ids = set(
    ranking_sessions[split_idx:]
)

rank_train = ranking_df[
    ranking_df["session"].isin(
        train_session_ids
    )
].copy()

rank_valid = ranking_df[
    ranking_df["session"].isin(
        valid_session_ids
    )
].copy()

print(rank_train.shape)
print(rank_valid.shape)


## 9. Prepare LambdaMART groups

In [ ]:
FEATURE_COLUMNS = [
    "session_length",
    "candidate_seen_before",
    "item_interactions",
    "item_clicks",
    "item_carts",
    "item_orders",
    "item_cart_rate",
    "item_order_rate",
    "covisit_rank",
    "covisit_rr",
    "item2vec_rank",
    "item2vec_rr",
    "bpr_rank",
    "bpr_rr",
    "two_tower_rank",
    "two_tower_rr",
]

rank_train = rank_train.sort_values(
    "session"
)

rank_valid = rank_valid.sort_values(
    "session"
)

X_train = rank_train[FEATURE_COLUMNS]
y_train = rank_train["label"]

X_valid = rank_valid[FEATURE_COLUMNS]
y_valid = rank_valid["label"]

train_groups = (
    rank_train
    .groupby("session", sort=False)
    .size()
    .to_numpy()
)

valid_groups = (
    rank_valid
    .groupby("session", sort=False)
    .size()
    .to_numpy()
)

print(f"Train groups: {len(train_groups):,}")
print(f"Valid groups: {len(valid_groups):,}")


## 10. Train LightGBM LambdaMART

`lambdarank` is a learning-to-rank objective. It optimizes ordering inside each query/session group rather than treating candidate rows independently.


In [ ]:
ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_SEED,
)

ranker.fit(
    X_train,
    y_train,
    group=train_groups,
    eval_set=[
        (X_valid, y_valid)
    ],
    eval_group=[
        valid_groups
    ],
    eval_at=[5, 10, 20],
    callbacks=[
        lgb.early_stopping(50),
        lgb.log_evaluation(50),
    ],
)


## 11. Evaluate final ranking

In [ ]:
rank_valid["score"] = ranker.predict(
    rank_valid[FEATURE_COLUMNS]
)

def evaluate_ranked_dataframe(
    df,
    k=20,
):
    hits = []
    reciprocal_ranks = []

    for _, group in df.groupby(
        "session",
        sort=False,
    ):
        ranked = (
            group.sort_values(
                "score",
                ascending=False,
            )
            .head(k)
        )

        labels = ranked[
            "label"
        ].to_numpy()

        if labels.any():
            rank = (
                np.argmax(labels)
                + 1
            )

            hits.append(1.0)
            reciprocal_ranks.append(
                1.0 / rank
            )
        else:
            hits.append(0.0)
            reciprocal_ranks.append(0.0)

    return {
        f"Recall@{k}": float(
            np.mean(hits)
        ),
        f"MRR@{k}": float(
            np.mean(reciprocal_ranks)
        ),
    }


evaluate_ranked_dataframe(
    rank_valid,
    k=20,
)


## 12. Inspect feature importance

In [ ]:
feature_importance = (
    pd.Series(
        ranker.feature_importances_,
        index=FEATURE_COLUMNS,
    )
    .sort_values(
        ascending=False
    )
)

feature_importance


## 13. Save the ranker

In [ ]:
ranker.booster_.save_model(
    str(
        MODEL_DIR
        / "lambdamart_ranker.txt"
    )
)

feature_importance.to_csv(
    PROCESSED_DIR
    / "ranker_feature_importance.csv",
    header=["importance"],
)

print("Saved ranking model.")


## Next modeling step

The retrieval + ranking architecture is now structurally complete.

The next notebook should cover sequence-aware deep recommenders:

- GRU4Rec
- SASRec
- a stronger modern variant such as TRON

Those models can be evaluated both as next-item predictors and as additional candidate generators before we freeze the local architecture and begin productionization.
